#### LLM Integration

Why LLM Integration Is a Dedicated Module

In every module so far, we used ChatOpenAI as the default LLM. But in real production systems:

* Clients may mandate Azure OpenAI for data residency compliance
* Some tasks need Anthropic Claude for long-context reasoning
* Cost-sensitive tasks should use Gemini Flash or a smaller model
* Regulated industries may need on-premise or private models
* A single agent may need to switch models mid-graph based on task complexity


LangGraph is model-agnostic. Every LLM it supports exposes the same invoke(), stream(), and bind_tools() interface — so swapping providers requires changing one line, not rewriting your graph.

#### The Unified Interface — Why It Works

All LangChain-compatible LLMs extend BaseChatModel. This means:

``` python
# These all work identically in a graph node:
llm.invoke(messages)
llm.stream(messages)
llm.bind_tools(tools)
llm.with_structured_output(schema)
```

The graph node does not care which provider is underneath. You can swap providers without touching graph logic.

#### Setup — Install All Providers

``` python
pip install langchain-openai        # OpenAI + Azure OpenAI
pip install langchain-anthropic     # Anthropic Claude
pip install langchain-google-genai  # Google Gemini
pip install langchain-groq # groq
pip install langchain-community     # Misc community providers
pip install python-dotenv           # For loading .env files
```

``` python
# .env file
OPENAI_API_KEY=sk-...
AZURE_OPENAI_API_KEY=...
AZURE_OPENAI_ENDPOINT=https://your-resource.openai.azure.com/
ANTHROPIC_API_KEY=sk-ant-...
GOOGLE_API_KEY=AIza...
```

``` python
from dotenv import load_dotenv
load_dotenv()
```

Part 1 — OpenAI

1.1 Basic Setup



In [ ]:
from langchain_openai import ChatOpenAI

# Minimal — uses OPENAI_API_KEY from environment
llm = ChatOpenAI(model="gpt-4o-mini")

# Full configuration
llm = ChatOpenAI(
    model="gpt-4o-mini",          # or "gpt-4o", "gpt-4-turbo", "o1-mini"
    temperature=0.0,               # 0 = deterministic, 1 = creative
    max_tokens=2048,               # max output tokens
    timeout=30,                    # request timeout in seconds
    max_retries=3,                 # auto-retry on rate limit / server errors
    api_key="sk-...",              # or set OPENAI_API_KEY env var
)

1.2 Available OpenAI Models

``` python
# Fast and cheap — everyday agent tasks
gpt4o_mini = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Powerful — complex reasoning, long context
gpt4o = ChatOpenAI(model="gpt-4o", temperature=0)

# Reasoning model — math, coding, multi-step problems (no temperature control)
o1_mini = ChatOpenAI(model="o1-mini")

# Latest flagship
gpt4o_latest = ChatOpenAI(model="gpt-4o-2024-11-20", temperature=0)
```

1.3 Using OpenAI in a LangGraph Node



In [ ]:
from langgraph.graph import StateGraph, MessagesState, END
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool

@tool
def get_product_price(product_name: str) -> str:
    """Get price of a product from the catalog.
    
    Args:
        product_name: Name of the product to look up
    """
    catalog = {
        "laptop": "₹75,000",
        "phone": "₹25,000",
        "tablet": "₹35,000"
    }
    return catalog.get(product_name.lower(), f"Product '{product_name}' not found")

tools = [get_product_price]
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
llm_with_tools = llm.bind_tools(tools)

SYSTEM_PROMPT = "You are a helpful product catalog assistant."

def openai_node(state: MessagesState) -> dict:
    messages = [SystemMessage(content=SYSTEM_PROMPT)] + state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

tool_node = ToolNode(tools)

builder = StateGraph(MessagesState)
builder.add_node("llm", openai_node)
builder.add_node("tools", tool_node)
builder.set_entry_point("llm")
builder.add_conditional_edges("llm", tools_condition)
builder.add_edge("tools", "llm")
builder.add_edge("llm", END)  # when no tools, end

graph = builder.compile()

result = graph.invoke({
    "messages": [HumanMessage(content="What is the price of a laptop?")]
})
print(result["messages"][-1].content)

1.4 Structured Output with OpenAI



In [ ]:
from pydantic import BaseModel, Field
from typing import List

class ProductRecommendation(BaseModel):
    product_name: str = Field(description="Name of the recommended product")
    price_inr: int = Field(description="Price in Indian Rupees")
    reason: str = Field(description="Why this product is recommended")
    confidence: float = Field(description="Confidence score between 0 and 1")

class RecommendationList(BaseModel):
    recommendations: List[ProductRecommendation]
    summary: str

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
structured_llm = llm.with_structured_output(RecommendationList)

result = structured_llm.invoke(
    "Recommend 2 products for a student on a tight budget"
)

print(f"Summary: {result.summary}")
for rec in result.recommendations:
    print(f"  {rec.product_name} — ₹{rec.price_inr:,} (confidence: {rec.confidence})")
    print(f"  Reason: {rec.reason}")

1.5 Streaming OpenAI Responses

``` python
llm = ChatOpenAI(model="gpt-4o-mini", streaming=True)

print("Streaming response: ", end="", flush=True)
for chunk in llm.stream("Explain LangGraph in 3 sentences"):
    print(chunk.content, end="", flush=True)
print()  # newline after stream completes
```

Part 2 — Azure OpenAI

2.1 Why Azure OpenAI

Azure OpenAI runs OpenAI models (GPT-4o, GPT-4, etc.) inside Microsoft's cloud infrastructure. Enterprises choose it for:

* Data residency — data stays within a specific region
* Private networking — no traffic over public internet
* Enterprise SLAs — higher availability guarantees
* Compliance — SOC2, HIPAA, GDPR compliance built-in

The models are identical to OpenAI — only the endpoint and auth differ.

2.2 Azure OpenAI Setup

Step 1 — Create a resource in Azure Portal

``` markdown
Azure Portal → Create Resource → Azure OpenAI
Region: East US / West Europe / Southeast Asia
Pricing tier: Standard S0
```

Step 2 — Deploy a model

``` markdown
Azure OpenAI Studio → Deployments → Create Deployment
Model: gpt-4o-mini
Deployment name: gpt-4o-mini-prod  ← this is your deployment_name
```

Step 3 — Get credentials

``` markdown
Azure OpenAI resource → Keys and Endpoint
AZURE_OPENAI_API_KEY = Key 1 value
AZURE_OPENAI_ENDPOINT = https://your-resource.openai.azure.com/
```

2.3 Using Azure OpenAI in LangGraph



In [ ]:
from langchain_openai import AzureChatOpenAI
import os

llm = AzureChatOpenAI(
    azure_deployment="gpt-4o-mini-prod",     # your deployment name in Azure
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version="2024-08-01-preview",        # Azure API version
    temperature=0,
    max_tokens=2048,
    max_retries=3,
)

# From here — exactly the same as OpenAI
# bind_tools, with_structured_output, stream — all work identically

response = llm.invoke("What is the capital of Karnataka?")
print(response.content)

2.4 Azure OpenAI in a Graph Node



In [ ]:
from langchain_openai import AzureChatOpenAI
from langgraph.graph import StateGraph, MessagesState, END
from langchain_core.messages import HumanMessage, SystemMessage

azure_llm = AzureChatOpenAI(
    azure_deployment="gpt-4o-mini-prod",
    api_version="2024-08-01-preview",
    temperature=0
)

def azure_llm_node(state: MessagesState) -> dict:
    messages = [
        SystemMessage(content="You are a corporate assistant. Be concise and professional.")
    ] + state["messages"]
    response = azure_llm.invoke(messages)
    return {"messages": [response]}

builder = StateGraph(MessagesState)
builder.add_node("llm", azure_llm_node)
builder.set_entry_point("llm")
builder.add_edge("llm", END)

graph = builder.compile()

result = graph.invoke({
    "messages": [HumanMessage(content="Summarize the key benefits of Azure OpenAI for enterprise use")]
})
print(result["messages"][-1].content)

2.6 Azure OpenAI — Common Errors and Fixes

``` python
# Error: AuthenticationError
# Fix: Check AZURE_OPENAI_API_KEY and AZURE_OPENAI_ENDPOINT are correct

# Error: ResourceNotFound — DeploymentNotFound
# Fix: azure_deployment must match exact deployment name in Azure Studio

# Error: InvalidRequestError — API version mismatch
# Fix: Use latest api_version: "2024-08-01-preview"

# Error: RateLimitError
# Fix: Use max_retries=3 and add exponential backoff
azure_llm = AzureChatOpenAI(
    azure_deployment="gpt-4o-mini-prod",
    api_version="2024-08-01-preview",
    max_retries=3,       # auto-retry on rate limits
    timeout=30,
    temperature=0
)
```

Part 3 — Anthropic Claude

3.1 Why Anthropic Claude

Claude excels at:

* Long context — Claude 3.5 Sonnet handles 200K token context windows
* Nuanced instruction following — better at complex multi-part instructions
* Safety and refusal calibration — less likely to refuse legitimate requests
* Document analysis — excellent at reading and reasoning over large documents
* Code generation — strong coding performance especially for Python

3.2 Setup



In [ ]:
from langchain_anthropic import ChatAnthropic

# Minimal
llm = ChatAnthropic(model="claude-3-5-sonnet-20241022")

# Full configuration
llm = ChatAnthropic(
    model="claude-3-5-sonnet-20241022",
    temperature=0,
    max_tokens=4096,
    timeout=60,
    max_retries=3,
    api_key=os.getenv("ANTHROPIC_API_KEY")
)

3.3 Available Claude Models

``` python
# Fast + cheap — high-volume tasks, classification, extraction
claude_haiku = ChatAnthropic(model="claude-3-5-haiku-20241022", temperature=0)

# Balanced — most tasks, recommended default
claude_sonnet = ChatAnthropic(model="claude-3-5-sonnet-20241022", temperature=0)

# Most powerful — complex reasoning, research, long documents
claude_opus = ChatAnthropic(model="claude-3-opus-20240229", temperature=0)

Part 4 — Google Gemini

4.1 Why Gemini

Gemini's strengths:


* Multimodal — natively handles text, images, audio, video in one model
* Gemini Flash — extremely fast and cheap, great for high-throughput tasks
* Long context — Gemini 1.5 Pro supports 1 million token context
* Google ecosystem — native integration with Google Search, Workspace, BigQuery
* Cost — Gemini Flash is among the cheapest capable models available


4.2 Setup

``` python
from langchain_google_genai import ChatGoogleGenerativeAI

# Minimal
llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash")

# Full configuration
llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    temperature=0,
    max_output_tokens=2048,
    google_api_key=os.getenv("GOOGLE_API_KEY"),
    convert_system_message_to_human=True  # Gemini needs this for system messages
)

Part 5 — Using Multiple LLMs in One Graph

5.1 Why Multiple LLMs

A single agent workflow might benefit from using different models for different steps:

| Step | Task | Best Model | Reason |
|------|------|------------|--------|
| Intent classification | Simple routing | Gemini Flash | Fast + cheap |
| Document analysis | Long context | Claude Sonnet | 200K context |
| Code generation | Writing code | GPT-4o | Strong coding |
| Final formatting | Polish output | GPT-4o-mini | Good quality, cheap |

5.2 Multiple LLMs — Different Nodes, Different Models

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, END
from typing import TypedDict, List

class PipelineState(TypedDict):
    user_input: str
    intent: str
    analysis: str
    code: str
    final_output: str

# --- Initialize models ---
gemini_flash = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    temperature=0,
    convert_system_message_to_human=True
)
claude_sonnet = ChatAnthropic(
    model="claude-3-5-sonnet-20241022",
    temperature=0
)
gpt4o = ChatOpenAI(model="gpt-4o", temperature=0)
gpt4o_mini = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# --- Node 1: Classify intent with fast/cheap Gemini Flash ---
def classify_intent(state: PipelineState) -> dict:
    prompt = f"""Classify this request into one of: 
    [data_analysis, code_generation, document_review, general_query]
    
    Request: {state['user_input']}
    
    Respond with only the category name."""
    
    from langchain_core.messages import HumanMessage
    response = gemini_flash.invoke([HumanMessage(content=prompt)])
    intent = response.content.strip().lower()
    print(f"[classify_intent] Intent: {intent} (used: Gemini Flash)")
    return {"intent": intent}

# --- Node 2: Deep analysis with Claude's long context ---
def analyze_with_claude(state: PipelineState) -> dict:
    from langchain_core.messages import HumanMessage, SystemMessage
    
    messages = [
        SystemMessage(content="You are a senior analyst. Provide thorough, structured analysis."),
        HumanMessage(content=f"Analyze this thoroughly: {state['user_input']}")
    ]
    response = claude_sonnet.invoke(messages)
    print(f"[analyze_with_claude] Analysis done (used: Claude Sonnet)")
    return {"analysis": response.content}

# --- Node 3: Code generation with GPT-4o ---
def generate_code(state: PipelineState) -> dict:
    from langchain_core.messages import HumanMessage, SystemMessage
    
    messages = [
        SystemMessage(content="You are an expert Python developer. Write clean, production-ready code."),
        HumanMessage(content=f"Write Python code for: {state['user_input']}")
    ]
    response = gpt4o.invoke(messages)
    print(f"[generate_code] Code generated (used: GPT-4o)")
    return {"code": response.content}

# --- Node 4: Format final output with cheap GPT-4o-mini ---
def format_output(state: PipelineState) -> dict:
    from langchain_core.messages import HumanMessage
    
    content = state.get("analysis") or state.get("code") or "No content generated"
    
    prompt = f"""Format this content into a clean, professional response:

{content}

Make it concise, well-structured, and easy to read."""
    
    response = gpt4o_mini.invoke([HumanMessage(content=prompt)])
    print(f"[format_output] Formatted (used: GPT-4o-mini)")
    return {"final_output": response.content}

# --- Routing ---
def route_by_intent(state: PipelineState) -> str:
    intent = state.get("intent", "general_query")
    if intent in ["data_analysis", "document_review"]:
        return "analyze"
    elif intent == "code_generation":
        return "code"
    else:
        return "format"  # general queries go straight to formatting

# --- Build graph ---
builder = StateGraph(PipelineState)
builder.add_node("classify", classify_intent)
builder.add_node("analyze", analyze_with_claude)
builder.add_node("code", generate_code)
builder.add_node("format", format_output)

builder.set_entry_point("classify")
builder.add_conditional_edges(
    "classify",
    route_by_intent,
    {
        "analyze": "analyze",
        "code": "code",
        "format": "format"
    }
)
builder.add_edge("analyze", "format")
builder.add_edge("code", "format")
builder.add_edge("format", END)

graph = builder.compile()

# --- Test ---
test_inputs = [
    "Write a Python function to parse JSON files and load them into a pandas DataFrame",
    "Analyze the trade-offs between microservices and monolithic architecture",
    "What is the difference between PRIMARY KEY and UNIQUE KEY in SQL?"
]

for user_input in test_inputs:
    print(f"\n{'='*60}")
    print(f"Input: {user_input}")
    print("="*60)
    result = graph.invoke({"user_input": user_input})
    print(f"\nFinal Output:\n{result['final_output']}")

5.3 Fallback Chain — Primary + Backup Model

If the primary model is unavailable or over rate-limit, automatically fall back to another:



In [ ]:
from langchain_core.language_models import BaseChatModel
from langchain_core.messages import BaseMessage
from typing import List

class FallbackLLM:
    """Wraps multiple LLMs with automatic fallback."""
    
    def __init__(self, primary: BaseChatModel, fallback: BaseChatModel):
        self.primary = primary
        self.fallback = fallback
    
    def invoke(self, messages: List[BaseMessage]):
        try:
            response = self.primary.invoke(messages)
            print(f"[FallbackLLM] Used primary model")
            return response
        except Exception as e:
            print(f"[FallbackLLM] Primary failed ({e}). Using fallback.")
            return self.fallback.invoke(messages)
    
    def bind_tools(self, tools):
        """Bind tools to both models."""
        self.primary = self.primary.bind_tools(tools)
        self.fallback = self.fallback.bind_tools(tools)
        return self

# Usage
resilient_llm = FallbackLLM(
    primary=ChatOpenAI(model="gpt-4o", temperature=0),
    fallback=ChatAnthropic(model="claude-3-5-sonnet-20241022", temperature=0)
)

# Or use LangChain's built-in fallback method
primary = ChatOpenAI(model="gpt-4o", temperature=0)
fallback = ChatAnthropic(model="claude-3-5-sonnet-20241022", temperature=0)
resilient_llm_native = primary.with_fallbacks([fallback])

# Use in a node
def resilient_node(state: MessagesState) -> dict:
    response = resilient_llm_native.invoke(state["messages"])
    return {"messages": [response]}

Part 6 — Dynamic Model Selection

Dynamic model selection means choosing which LLM to use at runtime, based on task properties like complexity, required context length, cost budget, or user tier.

6.1 Strategy 1 — Select by Task Complexity



In [ ]:
from typing import TypedDict, Literal
from langchain_core.messages import HumanMessage, AIMessage

class TaskState(TypedDict):
    user_input: str
    complexity: Literal["simple", "moderate", "complex"]
    model_used: str
    output: str

# Pre-initialized models (initialize once, reuse)
MODELS = {
    "simple": ChatGoogleGenerativeAI(
        model="gemini-1.5-flash",
        temperature=0,
        convert_system_message_to_human=True
    ),
    "moderate": ChatOpenAI(model="gpt-4o-mini", temperature=0),
    "complex": ChatAnthropic(model="claude-3-5-sonnet-20241022", temperature=0)
}

def assess_complexity(state: TaskState) -> dict:
    """Use a cheap model to classify task complexity."""
    
    prompt = f"""Rate the complexity of this task as simple, moderate, or complex.

Simple: factual questions, basic definitions, simple calculations
Moderate: multi-step reasoning, comparisons, summaries
Complex: deep analysis, long document processing, research synthesis

Task: {state['user_input']}

Respond with only one word: simple, moderate, or complex"""
    
    # Always use cheap model for classification
    classifier = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    response = classifier.invoke([HumanMessage(content=prompt)])
    complexity = response.content.strip().lower()
    
    if complexity not in ["simple", "moderate", "complex"]:
        complexity = "moderate"  # safe default
    
    print(f"[assess_complexity] Task complexity: {complexity}")
    return {"complexity": complexity}

def execute_with_selected_model(state: TaskState) -> dict:
    """Execute task using the model selected based on complexity."""
    
    complexity = state["complexity"]
    selected_model = MODELS[complexity]
    model_name = {
        "simple": "Gemini 1.5 Flash",
        "moderate": "GPT-4o-mini",
        "complex": "Claude 3.5 Sonnet"
    }[complexity]
    
    print(f"[execute] Using {model_name} for {complexity} task")
    
    response = selected_model.invoke([
        HumanMessage(content=state["user_input"])
    ])
    
    return {
        "output": response.content,
        "model_used": model_name
    }

builder = StateGraph(TaskState)
builder.add_node("assess", assess_complexity)
builder.add_node("execute", execute_with_selected_model)
builder.set_entry_point("assess")
builder.add_edge("assess", "execute")
builder.add_edge("execute", END)

dynamic_graph = builder.compile()

# Test
test_tasks = [
    "What is 15% of 8000?",
    "Compare REST and GraphQL APIs for a mobile application",
    "Analyze the architectural trade-offs of building a real-time data pipeline using Kafka vs. Kinesis for a fintech company processing 10 million transactions per day"
]

for task in test_tasks:
    print(f"\nTask: {task[:60]}...")
    result = dynamic_graph.invoke({"user_input": task})
    print(f"Model used: {result['model_used']}")
    print(f"Output: {result['output'][:200]}...")

6.2 Strategy 2 — Select by User Tier

Different subscription tiers get different model quality:




In [ ]:
from typing import Optional

class TieredState(TypedDict):
    user_input: str
    user_tier: Literal["free", "pro", "enterprise"]
    output: str
    model_used: str

TIER_MODELS = {
    "free": {
        "model": ChatGoogleGenerativeAI(
            model="gemini-1.5-flash",
            temperature=0.3,
            convert_system_message_to_human=True
        ),
        "name": "Gemini 1.5 Flash",
        "max_tokens": 500
    },
    "pro": {
        "model": ChatOpenAI(model="gpt-4o-mini", temperature=0),
        "name": "GPT-4o-mini",
        "max_tokens": 2000
    },
    "enterprise": {
        "model": ChatAnthropic(model="claude-3-5-sonnet-20241022", temperature=0),
        "name": "Claude 3.5 Sonnet",
        "max_tokens": 8000
    }
}

def tiered_llm_node(state: TieredState) -> dict:
    tier = state.get("user_tier", "free")
    tier_config = TIER_MODELS[tier]
    
    print(f"[tiered_llm_node] Tier: {tier} → Model: {tier_config['name']}")
    
    response = tier_config["model"].invoke([
        HumanMessage(content=state["user_input"])
    ])
    
    return {
        "output": response.content,
        "model_used": tier_config["name"]
    }

builder = StateGraph(TieredState)
builder.add_node("llm", tiered_llm_node)
builder.set_entry_point("llm")
builder.add_edge("llm", END)

tiered_graph = builder.compile()

# Simulate different tier users
users = [
    {"user_input": "Explain LangGraph", "user_tier": "free"},
    {"user_input": "Explain LangGraph", "user_tier": "pro"},
    {"user_input": "Explain LangGraph", "user_tier": "enterprise"}
]

for user in users:
    result = tiered_graph.invoke(user)
    print(f"\nTier: {user['user_tier']} → Used: {result['model_used']}")

6.3 Strategy 3 — Select by Context Length

Automatically route to a long-context model when the input is large:



In [ ]:
def select_model_by_context_length(text: str) -> tuple:
    """
    Select the appropriate model based on input token estimate.
    Returns (model, model_name).
    """
    # Approximate: 1 token ≈ 4 characters
    estimated_tokens = len(text) / 4
    
    if estimated_tokens < 4000:
        # Short context — use fast, cheap model
        return (
            ChatOpenAI(model="gpt-4o-mini", temperature=0),
            "GPT-4o-mini (short context)"
        )
    elif estimated_tokens < 50000:
        # Medium context — GPT-4o handles up to 128K
        return (
            ChatOpenAI(model="gpt-4o", temperature=0),
            "GPT-4o (medium context)"
        )
    else:
        # Long context — Claude 200K or Gemini 1M
        return (
            ChatAnthropic(model="claude-3-5-sonnet-20241022", temperature=0),
            "Claude 3.5 Sonnet (long context)"
        )

class ContextAwareState(TypedDict):
    document: str
    question: str
    model_used: str
    answer: str

def context_aware_node(state: ContextAwareState) -> dict:
    full_input = state["document"] + state["question"]
    model, model_name = select_model_by_context_length(full_input)
    
    print(f"[context_aware_node] Input ~{len(full_input)//4} tokens → {model_name}")
    
    prompt = f"Document:\n{state['document']}\n\nQuestion: {state['question']}"
    response = model.invoke([HumanMessage(content=prompt)])
    
    return {"answer": response.content, "model_used": model_name}

builder = StateGraph(ContextAwareState)
builder.add_node("answer", context_aware_node)
builder.set_entry_point("answer")
builder.add_edge("answer", END)

context_graph = builder.compile()

6.4 Strategy 4 — Cost-Aware Selection with Budget Tracking

In [ ]:
# Approximate costs per 1000 tokens (as of early 2025)
MODEL_COSTS = {
    "gemini-1.5-flash":              {"input": 0.000075, "output": 0.0003},
    "gpt-4o-mini":                   {"input": 0.00015,  "output": 0.0006},
    "gpt-4o":                        {"input": 0.0025,   "output": 0.01},
    "claude-3-5-sonnet-20241022":    {"input": 0.003,    "output": 0.015},
    "claude-3-opus-20240229":        {"input": 0.015,    "output": 0.075},
}

class BudgetAwareState(TypedDict):
    user_input: str
    budget_usd: float           # max spend for this request
    output: str
    model_used: str
    estimated_cost_usd: float

def budget_aware_node(state: BudgetAwareState) -> dict:
    budget = state.get("budget_usd", 0.01)
    input_tokens_estimate = len(state["user_input"]) / 4
    output_tokens_estimate = 500  # assume 500 output tokens
    
    selected_model = None
    selected_name = None
    selected_cost = 0
    
    # Try models from cheapest to most expensive
    model_preference = [
        ("gemini-1.5-flash", ChatGoogleGenerativeAI(
            model="gemini-1.5-flash", temperature=0, convert_system_message_to_human=True)),
        ("gpt-4o-mini", ChatOpenAI(model="gpt-4o-mini", temperature=0)),
        ("gpt-4o", ChatOpenAI(model="gpt-4o", temperature=0)),
        ("claude-3-5-sonnet-20241022", ChatAnthropic(
            model="claude-3-5-sonnet-20241022", temperature=0)),
    ]
    
    for model_id, model_obj in model_preference:
        costs = MODEL_COSTS[model_id]
        estimated_cost = (
            (input_tokens_estimate / 1000) * costs["input"] +
            (output_tokens_estimate / 1000) * costs["output"]
        )
        
        if estimated_cost <= budget:
            selected_model = model_obj
            selected_name = model_id
            selected_cost = estimated_cost
    
    if not selected_model:
        return {
            "output": "Budget too low for any available model.",
            "model_used": "none",
            "estimated_cost_usd": 0
        }
    
    print(f"[budget_aware] Budget: ${budget:.4f} → Selected: {selected_name} (est. ${selected_cost:.6f})")
    
    response = selected_model.invoke([HumanMessage(content=state["user_input"])])
    
    return {
        "output": response.content,
        "model_used": selected_name,
        "estimated_cost_usd": selected_cost
    }

Part 7 — Model Configuration Best Practices

7.1 Centralize Model Configuration

Never scatter model instantiations across nodes. Define them once:



In [ ]:
# config/models.py

import os
from langchain_openai import ChatOpenAI, AzureChatOpenAI
from langchain_anthropic import ChatAnthropic
from langchain_google_genai import ChatGoogleGenerativeAI

class ModelRegistry:
    """Single source of truth for all LLM configurations."""
    
    _models = {}  # cache initialized models
    
    @classmethod
    def get(cls, model_key: str):
        if model_key not in cls._models:
            cls._models[model_key] = cls._initialize(model_key)
        return cls._models[model_key]
    
    @classmethod
    def _initialize(cls, key: str):
        configs = {
            "fast": lambda: ChatGoogleGenerativeAI(
                model="gemini-1.5-flash", temperature=0,
                convert_system_message_to_human=True
            ),
            "standard": lambda: ChatOpenAI(
                model="gpt-4o-mini", temperature=0, max_retries=3
            ),
            "powerful": lambda: ChatOpenAI(
                model="gpt-4o", temperature=0, max_retries=3
            ),
            "reasoning": lambda: ChatAnthropic(
                model="claude-3-5-sonnet-20241022", temperature=0, max_retries=3
            ),
            "azure_standard": lambda: AzureChatOpenAI(
                azure_deployment=os.getenv("AZURE_DEPLOYMENT_NAME", "gpt-4o-mini"),
                api_version="2024-08-01-preview",
                temperature=0, max_retries=3
            ),
        }
        
        if key not in configs:
            raise ValueError(f"Unknown model key: '{key}'. Available: {list(configs.keys())}")
        
        return configs[key]()

# Usage in any node:
def my_node(state):
    llm = ModelRegistry.get("standard")
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

7.2 Environment-Based Model Switching



In [ ]:
import os

def get_llm_for_environment():
    """Return the right model based on the environment."""
    
    env = os.getenv("APP_ENV", "development")
    
    if env == "production":
        # Production: use best available, Azure for compliance
        return AzureChatOpenAI(
            azure_deployment="gpt-4o-prod",
            api_version="2024-08-01-preview",
            temperature=0,
            max_retries=3
        )
    elif env == "staging":
        # Staging: same model class as prod, cheaper model
        return ChatOpenAI(model="gpt-4o-mini", temperature=0)
    else:
        # Development: cheapest possible
        return ChatGoogleGenerativeAI(
            model="gemini-1.5-flash",
            temperature=0,
            convert_system_message_to_human=True
        )

# In your graph:
llm = get_llm_for_environment()

7.3 Model Metadata Tracking in State

Always track which model was used — critical for debugging and cost auditing:

In [ ]:
from typing import TypedDict, List,Annotated
import operator
from datetime import datetime

class TrackedState(TypedDict):
    messages: Annotated[List, operator.add]
    model_usage_log: Annotated[List[dict], operator.add]

def tracked_llm_node(state: TrackedState, model_key: str = "standard") -> dict:
    llm = ModelRegistry.get(model_key)
    
    start_time = datetime.now()
    response = llm.invoke(state["messages"])
    elapsed_ms = (datetime.now() - start_time).total_seconds() * 1000
    
    log_entry = {
        "timestamp": start_time.isoformat(),
        "model_key": model_key,
        "node": "llm_node",
        "latency_ms": round(elapsed_ms, 2),
        "input_chars": sum(len(str(m.content)) for m in state["messages"]),
        "output_chars": len(response.content)
    }
    
    return {
        "messages": [response],
        "model_usage_log": [log_entry]
    }

Part 9 — Full Multi-Provider Agent (Production Pattern)

In [ ]:
"""
Production-ready multi-provider agent that:
- Classifies task type (cheap model)
- Routes to the best model for the task
- Falls back on failure
- Tracks usage for cost monitoring
"""

import os
from typing import TypedDict, Literal, List, Annotated
import operator
from datetime import datetime

from langchain_openai import ChatOpenAI # type:ignore
from langchain_anthropic import ChatAnthropic # type:ignore
from langchain_google_genai import ChatGoogleGenerativeAI # type:ignore
from langchain_core.messages import HumanMessage, SystemMessage, BaseMessage # type:ignore
from langchain_core.tools import tool # type:ignore
from langgraph.graph import StateGraph, MessagesState, END # type:ignore
from langgraph.prebuilt import ToolNode, tools_condition # type:ignore
from langgraph.checkpoint.memory import MemorySaver # type:ignore

# ── State ────────────────────────────────────────────────────────────────────

class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], operator.add]
    task_type: str
    model_used: str
    usage_log: Annotated[List[dict], operator.add]

# ── Tools ─────────────────────────────────────────────────────────────────────

@tool
def calculate(expression: str) -> str:
    """Evaluate a math expression. E.g. '1500 * 0.18'"""
    import math
    allowed = {"sqrt": math.sqrt, "pow": pow, "abs": abs, "round": round}
    try:
        return f"Result: {eval(expression, {'__builtins__': {}}, allowed)}"
    except Exception as e:
        return f"Error: {e}"

@tool
def get_current_date() -> str:
    """Get today's date."""
    return f"Today is {datetime.now().strftime('%B %d, %Y')}"

tools = [calculate, get_current_date]

# ── Models ────────────────────────────────────────────────────────────────────

gemini_flash = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash", temperature=0, convert_system_message_to_human=True)

gpt4o_mini = ChatOpenAI(model="gpt-4o-mini", temperature=0).with_fallbacks(
    [ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0,
                            convert_system_message_to_human=True)]
)

claude_sonnet = ChatAnthropic(model="claude-3-5-sonnet-20241022", temperature=0).with_fallbacks(
    [ChatOpenAI(model="gpt-4o", temperature=0)]
)

gpt4o_mini_with_tools = ChatOpenAI(model="gpt-4o-mini", temperature=0).bind_tools(tools)

# ── Nodes ─────────────────────────────────────────────────────────────────────

def classify_task(state: AgentState) -> dict:
    last_msg = state["messages"][-1].content
    prompt = f"""Classify this as one of: math_calc, deep_analysis, tool_use, general
    
Input: {last_msg}
Reply with only the category name."""
    
    response = gemini_flash.invoke([HumanMessage(content=prompt)])
    task_type = response.content.strip().lower()
    if task_type not in ["math_calc", "deep_analysis", "tool_use", "general"]:
        task_type = "general"
    
    print(f"[classify] → {task_type}")
    return {"task_type": task_type, "usage_log": [{"node": "classify", "model": "gemini-flash"}]}

def route_by_task(state: AgentState) -> str:
    return state["task_type"]

def math_node(state: AgentState) -> dict:
    response = gpt4o_mini_with_tools.invoke(state["messages"])
    return {"messages": [response], "model_used": "gpt-4o-mini", 
            "usage_log": [{"node": "math", "model": "gpt-4o-mini"}]}

def analysis_node(state: AgentState) -> dict:
    messages = [SystemMessage(content="You are a deep analytical thinker. Provide thorough analysis.")] \
               + state["messages"]
    response = claude_sonnet.invoke(messages)
    return {"messages": [response], "model_used": "claude-sonnet",
            "usage_log": [{"node": "analysis", "model": "claude-sonnet"}]}

def tool_use_node(state: AgentState) -> dict:
    response = gpt4o_mini_with_tools.invoke(state["messages"])
    return {"messages": [response], "model_used": "gpt-4o-mini-tools",
            "usage_log": [{"node": "tool_use", "model": "gpt-4o-mini"}]}

def general_node(state: AgentState) -> dict:
    response = gpt4o_mini.invoke(state["messages"])
    return {"messages": [response], "model_used": "gpt-4o-mini",
            "usage_log": [{"node": "general", "model": "gpt-4o-mini"}]}

tool_node = ToolNode(tools)

# ── Graph ─────────────────────────────────────────────────────────────────────

builder = StateGraph(AgentState)
builder.add_node("classify", classify_task)
builder.add_node("math", math_node)
builder.add_node("analysis", analysis_node)
builder.add_node("tool_use", tool_use_node)
builder.add_node("general", general_node)
builder.add_node("tools", tool_node)

builder.set_entry_point("classify")
builder.add_conditional_edges("classify", route_by_task, {
    "math_calc": "math",
    "deep_analysis": "analysis",
    "tool_use": "tool_use",
    "general": "general"
})
builder.add_conditional_edges("math", tools_condition)
builder.add_conditional_edges("tool_use", tools_condition)
builder.add_edge("tools", "tool_use")
builder.add_edge("analysis", END)
builder.add_edge("general", END)
builder.add_edge("math", END)

agent = builder.compile(checkpointer=MemorySaver())

# ── Test ──────────────────────────────────────────────────────────────────────

test_inputs = [
    "What is 18% GST on ₹45,000?",
    "Analyze the pros and cons of event-driven architecture for a large-scale data platform",
    "What is today's date?",
    "Explain what a Python decorator does"
]

for inp in test_inputs:
    print(f"\n{'='*60}\nQ: {inp}")
    config = {"configurable": {"thread_id": f"test-{inp[:10]}"}}
    result = agent.invoke(
        {"messages": [HumanMessage(content=inp)], "usage_log": []},
        config
    )
    print(f"Model used: {result.get('model_used', 'n/a')}")
    print(f"Answer: {result['messages'][-1].content[:300]}")